In [21]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing	import StandardScaler,LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

In [22]:
df=pd.read_csv("C:\\Users\\saian\\OneDrive\\Desktop\\ds\\ann\\Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
labelencoder=LabelEncoder()
df['Gender']=labelencoder.fit_transform(df['Gender'])
with open('labelencoder.pkl','wb') as f:
				pickle.dump(labelencoder,f)

In [24]:
onehotencoder=OneHotEncoder()
df2=onehotencoder.fit_transform(df[["Geography"]]).toarray()
with	open('onehotencoder.pkl','wb') as f:
				pickle.dump(onehotencoder,f)

In [25]:
names_geo=onehotencoder.get_feature_names_out(["Geography"])

In [26]:
names_geo

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [27]:
df=df.drop(["RowNumber","CustomerId","Surname","Geography"],axis=1)
df=pd.concat([df, pd.DataFrame(df2, columns=names_geo)], axis=1)

In [28]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [29]:
Xtrain, Xtest, Ytrain, Ytest = train_test_split(df.drop("Exited",axis=1), df["Exited"], test_size=0.2, random_state=42)

In [30]:
standardscaler=StandardScaler()
Xtrain=standardscaler.fit_transform(Xtrain)
Xtest=standardscaler.transform(Xtest)
with open('standardscaler.pkl','wb') as f:
				pickle.dump(standardscaler,f)

In [31]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [32]:
model=Sequential([
    Dense(64, activation='relu', input_shape=(Xtrain.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

c:\Users\saian\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)

In [35]:
import datetime
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [36]:
early_stopping=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [37]:
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

In [38]:
history=model.fit(Xtrain,Ytrain,validation_data=(Xtest,Ytest),epochs=100,batch_size=32,callbacks=[early_stopping,tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8355 - loss: 0.3972 - val_accuracy: 0.8575 - val_loss: 0.3563
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8528 - loss: 0.3520 - val_accuracy: 0.8585 - val_loss: 0.3401
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8562 - loss: 0.3472 - val_accuracy: 0.8565 - val_loss: 0.3430
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8618 - loss: 0.3412 - val_accuracy: 0.8575 - val_loss: 0.3438
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8625 - loss: 0.3393 - val_accuracy: 0.8590 - val_loss: 0.3377
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8608 - loss: 0.3368 - val_accuracy: 0.8610 - val_loss: 0.3393
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8614 - loss: 0.3344 - val_accuracy: 0.8540 - val_loss: 0.3501
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8606 - loss: 0.3333 - val_accu

In [39]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [40]:
%tensorboard --logdir logs/fit/b

Reusing TensorBoard on port 6006 (pid 23932), started 0:39:36 ago. (Use '!kill 23932' to kill it.)

In [41]:
model.save("ann_model.h5")

In [42]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
